# 1. 환경설정

In [1]:
import os
import requests
import pandas as pd
from datetime import datetime, timedelta
from google.cloud import bigquery

from datetime import datetime

# 2. 데이터 불러오기

In [ ]:
# 1. 설정
GCP_KEY_PATH = "gcp_key.json"
PROJECT_ID = "mosquito-pipeline"  # GCP 프로젝트 ID
DATASET_ID = "mosquito_db"
TABLE_ID = "fact_ai_mosquito_daily"
API_KEY = "71677872696879723132357869714565"

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = GCP_KEY_PATH

# 2. 5월 1일부터 오늘(10월 19일)까지 날짜 범위 생성
start_date = datetime(2026, 5, 1)
end_date = datetime(2026, 10, 19)

current_date = start_date
rows_list = []

print("과거 데이터(Backfill) 수집 시작...")

while current_date <= end_date:
    date_str = current_date.strftime("%Y-%m-%d")
    url = f"http://openapi.seoul.go.kr:8088/{API_KEY}/json/aiMosquitoStatus/1/5/{date_str}"
    
    try:
        response = requests.get(url)
        data = response.json()
        
        # 데이터가 정상적으로 들어온 경우만 추출
        if "aiMosquitoStatus" in data and "row" in data["aiMosquitoStatus"]:
            row = data["aiMosquitoStatus"]["row"][0]
            rows_list.append({
                "mosquito_date": row["MOSQUITO_DATE"],
                "mosquito_value_water": float(row["MOSQUITO_VALUE_WATER"]),
                "mosquito_value_house": float(row["MOSQUITO_VALUE_HOUSE"]),
                "mosquito_value_park": float(row["MOSQUITO_VALUE_PARK"]),
                "ta_max": float(row["TA_MAX"]),
                "ta_min": float(row["TA_MIN"]),
                "rn_day": float(row["RN_DAY"]),
                "created_at": datetime.now().isoformat()  # JSON 직렬화가 가능한 ISO 날짜 형식
            })
            print(f"[{date_str}] 수집 성공")
        else:
            print(f"[{date_str}] 데이터 없음/오류")
            
    except Exception as e:
        print(f"[{date_str}] API 호출 실패: {e}")
        
    current_date += timedelta(days=1)

# 3. BigQuery 일괄 적재 (pyarrow 없이 JSON 리스트로 직접 적재)
client = bigquery.Client(project=PROJECT_ID)
table_ref = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

print("\nBigQuery 적재 시작...")
# load_table_from_dataframe 대신 load_table_from_json 사용
job = client.load_table_from_json(rows_list, table_ref, job_config=job_config)
job.result()  # 작업 완료 대기

print(f"🎉 백필 완료! 총 {len(rows_list)}건의 데이터가 {table_ref} 에 저장되었습니다.")

과거 데이터(Backfill) 수집 시작...
[2026-05-01] 수집 성공
[2026-05-02] 수집 성공
[2026-05-03] 수집 성공
[2026-05-04] 수집 성공
[2026-05-05] 수집 성공
[2026-05-06] 수집 성공
[2026-05-07] 수집 성공
[2026-05-08] 수집 성공
[2026-05-09] 수집 성공
[2026-05-10] 수집 성공
[2026-05-11] 수집 성공
[2026-05-12] 수집 성공
[2026-05-13] 수집 성공
[2026-05-14] 수집 성공
[2026-05-15] 수집 성공
[2026-05-16] 수집 성공
[2026-05-17] 수집 성공
[2026-05-18] 수집 성공
[2026-05-19] 수집 성공
[2026-05-20] 수집 성공
[2026-05-21] 수집 성공
[2026-05-22] 수집 성공
[2026-05-23] 수집 성공
[2026-05-24] 수집 성공
[2026-05-25] 수집 성공
[2026-05-26] 수집 성공
[2026-05-27] 수집 성공
[2026-05-28] 수집 성공
[2026-05-29] 수집 성공
[2026-05-30] 수집 성공
[2026-05-31] 수집 성공
[2026-06-01] 수집 성공
[2026-06-02] 수집 성공
[2026-06-03] 수집 성공
[2026-06-04] 수집 성공
[2026-06-05] 수집 성공
[2026-06-06] 수집 성공
[2026-06-07] 수집 성공
[2026-06-08] 수집 성공
[2026-06-09] 수집 성공
[2026-06-10] 수집 성공
[2026-06-11] 수집 성공
[2026-06-12] 수집 성공
[2026-06-13] 수집 성공
[2026-06-14] 수집 성공
[2026-06-15] 수집 성공
[2026-06-16] 수집 성공
[2026-06-17] 수집 성공
[2026-06-18] 수집 성공
[2026-06-19] 수집 성공
[2026-06-20] 수집 성공
[2026

In [ ]:
# 1. 설정

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = GCP_KEY_PATH

# 2. 실행 시점 기준 '어제 날짜'를 'YYYY-MM-DD' 형식으로 자동 계산
yesterday_date = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
url = f"http://openapi.seoul.go.kr:8088/{API_KEY}/json/aiMosquitoStatus/1/5/{yesterday_date}"

print(f"[{yesterday_date}] 전날 모기예보 데이터 수집 시작...")

rows_list = []

try:
    response = requests.get(url)
    data = response.json()
    
    # 데이터가 정상적으로 들어온 경우만 추출
    if "aiMosquitoStatus" in data and "row" in data["aiMosquitoStatus"]:
        row = data["aiMosquitoStatus"]["row"][0]
        rows_list.append({
            "mosquito_date": row["MOSQUITO_DATE"],
            "mosquito_value_water": float(row["MOSQUITO_VALUE_WATER"]),
            "mosquito_value_house": float(row["MOSQUITO_VALUE_HOUSE"]),
            "mosquito_value_park": float(row["MOSQUITO_VALUE_PARK"]),
            "ta_max": float(row="TA_MAX"),
            "ta_min": float(row="TA_MIN"),
            "rn_day": float(row="RN_DAY"),
            "created_at": datetime.now().isoformat()
        })
        print(f"[{yesterday_date}] 데이터 파싱 성공")
    else:
        print(f"[{yesterday_date}] 데이터가 아직 업데이트되지 않았거나 없음/오류")

except Exception as e:
    print(f"[{yesterday_date}] API 호출 실패: {e}")

# 3. 데이터가 존재할 경우에만 BigQuery에 적재 (WRITE_APPEND 사용)
if rows_list:
    client = bigquery.Client(project=PROJECT_ID)
    table_ref = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND
    )

    job = client.load_table_from_json(rows_list, table_ref, job_config=job_config)
    job.result()

    print(f"🎉 [{yesterday_date}] 전날 일일 데이터 적재 완료!")
else:
    print(f"[{yesterday_date}] 적재할 데이터가 없어 작업을 스킵합니다.")

[2026-10-20] 데일리 모기예보 데이터 수집 시작...
API 응답 원본: {'RESULT': {'CODE': 'INFO-200', 'MESSAGE': '해당하는 데이터가 없습니다.'}}
[2026-10-20] 데이터가 아직 업데이트되지 않았거나 없음/오류
[2026-10-20] 적재할 데이터가 없어 작업을 스킵합니다.
